# Bancas do ICMC ao longo do ano: o padrão se repete?

**Pergunta:** como o volume de defesas e qualificações varia ao longo do ano, e esse padrão se repete entre os anos e difere entre programas, tipos (defesa/qualificação) e níveis (mestrado/doutorado)?

**Visualização:** gráfico sazonal de linhas (*cycle plot*).
- eixo X = semana do ano (ISO), eixo Y = nº de bancas na semana;
- **uma linha por ano**: se as linhas sobem e descem nas mesmas semanas, o padrão se repete;
- linha tracejada = média dos anos (só onde há pelo menos 2 anos com dados);
- semanas **sem e-mails não viram zero**: a linha quebra e uma faixa cinza marca o intervalo;
- anos parciais aparecem como "(parcial)" na legenda.

O notebook segue em ordem: carregar os dados, marcar semanas sem dados, montar as séries, desenhar o gráfico e, por fim, o painel interativo.

## 1. Instalação e importações
Usa `pandas`/`numpy` para os dados e `plotly` para o gráfico interativo. `ipywidgets` só é necessário para o painel da seção 8. Rode a primeira linha uma vez (pode apagar depois).

In [3]:
%pip install -q plotly ipywidgets pandas

# Ativa explicitamente o suporte do Colab para widgets de terceiros
from google.colab import output
output.enable_custom_widget_manager()

import html
import math
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 2. Configurações
Constantes usadas pelo gráfico: nomes dos meses para o eixo X, cores de cada ano, cor da faixa "sem dados", máximo de linhas no tooltip e a semana do ano em que cada mês começa (usada para rotular o eixo X).

In [4]:
MESES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
CORES_ANO = ["#8a97a8", "#2171b5", "#e6550d", "#31a354", "#756bb1", "#c51b8a"]
CINZA = "#c4c8cf"
MAX_LINHAS_TOOLTIP = 10
_SEMANAS = list(range(1, 54))
_TICKS = [int(pd.Timestamp(2025, m, 1).isocalendar().week) for m in range(1, 13)]

## 3. Carregar o CSV
Lê `eventos.csv` e cria duas colunas auxiliares:
- `segunda`: a segunda-feira da semana de cada banca (chave para agrupar por semana);
- `sigla`: só a sigla do programa (CCMC, PIPGES, ...), mais curta para filtros e títulos.

Linhas sem data são descartadas.

In [16]:
def carregar(caminho_csv):
    df = pd.read_csv(caminho_csv, parse_dates=["data"]).dropna(subset=["data"]).copy()
    df["segunda"] = df["data"] - pd.to_timedelta(df["data"].dt.weekday, unit="D")
    df["sigla"] = df["programa"].str.rsplit(" - ", n=1).str[-1]
    return df


# Ajustado para o caminho absoluto correto selecionado no ambiente
CAMINHO = "/content/eventos.csv"
df = carregar(CAMINHO)
print(len(df), "bancas,", df["data"].min().date(), "->", df["data"].max().date())
df.head()

1150 bancas, 2022-03-15 -> 2026-10-29


,id,tipo,nivel,programa,titulo,aluno,orientador,data,dia_semana,hora,local,icmc,remoto,atualizado,segunda,sigla
0,6524ce782f744a6d,Defesa,Doutorado,Ciências de Computação e Matemática Computacio...,Seis heurísticas de privacidade e usabilidade:...,André de Lima Salgado,Renata Pontin de Mattos Fortes,2022-03-15,1,09:00,NaN,NaN,1,1,2022-03-14,PPG-CCMC
1,d7e7717f6892bcc5,Qualificação,Doutorado,Estatística - PIPGES,Modelos de sobrevivência induzidos por fragili...,Michele Maciel Sacramento,Vicente Garibay Cancho,2022-03-15,1,14:00,NaN,NaN,1,1,2022-03-14,PIPGES
2,93cdcd8826dac3a0,Qualificação,Doutorado,Estatística - PIPGES,Flexible cloglog links for binary response in ...,Jessica Suzana Barragan Alves,Jorge Luis Bazán Guzmán,2022-03-15,1,14:00,NaN,NaN,1,1,2022-03-14,PIPGES
3,8253e00525ab1d05,Qualificação,Doutorado,Ciências de Computação e Matemática Computacio...,Continual Object Detection with Deep Neural Ne...,Ângelo Garangau Menezes,André Carlos Ponce de Leon Ferreira de Carvalho,2022-03-16,2,08:00,NaN,NaN,1,1,2022-03-14,PPG-CCMC
4,1cc8e08da6d26da3,Defesa,Mestrado,Matemática em Rede Nacional - PROFMAT,Investigações matemáticas como metodologia de ...,Breno Puertas de Freitas e Silva,Edna Maura Zuffi,2022-03-16,2,10:00,NaN,NaN,1,1,2022-03-14,PROFMAT


## 4. Semanas "sem dados" (não é zero!)
O CSV vem de e-mails, então há períodos em que **nenhum e-mail foi recebido** (ex.: março a início de maio de 2025). Contar essas semanas como 0 daria a impressão falsa de que não houve bancas.

`montar_grade` cria uma linha por semana entre a primeira e a última banca e marca como **sem dados** qualquer sequência de pelo menos `gap_min` semanas seguidas sem nenhuma banca (padrão: 6). Recessos curtos (dez/jan) continuam como zero real.

- Para mudar a regra: `montar_grade(df, gap_min=3)`.
- Para forçar um intervalo: `montar_grade(df, manual=[("2025-03-01", "2025-05-10")])`.
- Isso é calculado **antes dos filtros**; senão filtrar um programa pequeno criaria "buracos" falsos.

In [17]:
def montar_grade(df, gap_min=6, manual=()):
    """Uma linha por semana (segunda-feira) entre o 1º e o último evento.
    `gap` = sequência de >= gap_min semanas seguidas sem nenhuma banca (= sem e-mails, não zero).
    Recesso curto (dez/jan) continua zero. `manual`: [(inicio, fim), ...] força "sem dados".
    Calculado no df COMPLETO, antes dos filtros."""
    seg = pd.date_range(df["segunda"].min(), df["segunda"].max(), freq="7D")
    vazio = df.groupby("segunda").size().reindex(seg, fill_value=0).to_numpy() == 0
    gap = np.zeros(len(seg), bool)
    i = 0
    while i < len(seg):
        if vazio[i]:
            j = i
            while j < len(seg) and vazio[j]:
                j += 1
            if j - i >= gap_min:
                gap[i:j] = True
            i = j
        else:
            i += 1
    iso = seg.isocalendar()
    g = pd.DataFrame({"ano": iso["year"].astype(int).to_numpy(),
                      "semana": iso["week"].astype(int).to_numpy(), "gap": gap}, index=seg)
    for ini, fim in manual:
        g.loc[pd.Timestamp(ini):pd.Timestamp(fim), "gap"] = True
    return g


g = montar_grade(df)
print("semanas sem dados:", g.loc[g["gap"]].index.min().date(), "->", g.loc[g["gap"]].index.max().date(), f"({g['gap'].sum()} semanas)")
g.head()

semanas sem dados: NaT -> NaT (0 semanas)


,ano,semana,gap
2022-03-14,2022,11,False
2022-03-21,2022,12,False
2022-03-28,2022,13,False
2022-04-04,2022,14,False
2022-04-11,2022,15,False


## 5. Funções auxiliares das séries
- `_lista`: monta o texto do tooltip com as bancas da semana (data, hora, tipo/nível, aluno e programa; no máximo 10 linhas);
- `_semanal`: conta as bancas por (ano, semana do ano). Semanas sem dados, fora da cobertura ou fora do período escolhido ficam `NaN` (e não 0);
- `_suavizar`: média móvel centrada opcional (3 ou 5 semanas), que **não atravessa lacunas**;
- `_rotulo_ano`: acrescenta "(parcial)" no nome de anos que não cobrem a semana 1 a 52 (2024 começa em agosto; 2026 vai até outubro).

In [18]:
def _lista(d):
    if d is None or len(d) == 0:
        return ""
    d = d.sort_values(["data", "hora"], na_position="last")
    linhas = []
    for r in d.itertuples():
        h = f" {r.hora}" if isinstance(r.hora, str) else ""
        linhas.append(f"{r.data:%d/%m}{h} · {r.tipo}/{r.nivel} · {html.escape(str(r.aluno))} ({r.sigla})")
    extra = len(linhas) - MAX_LINHAS_TOOLTIP
    linhas = linhas[:MAX_LINHAS_TOOLTIP]
    if extra > 0:
        linhas.append(f"… e mais {extra}")
    return "<br>" + "<br>".join(linhas)


def _semanal(df, g, periodo=None):
    """Gera matrizes temporais de contagem e textos de tooltip de forma 100% segura.

    - Se a semana está num gap global do ano (sem e-mails), o valor é NaN (a linha quebra).
    - Se a semana está ativa, mas a categoria não possui eventos, o valor é exatamente 0.0."""
    anos = sorted(g["ano"].unique())
    ia = {a: k for k, a in enumerate(anos)}

    cont = np.full((len(anos), 53), np.nan)
    texto = np.full((len(anos), 53), "", dtype=object)

    ini, fim = periodo if periodo else (g.index[0], g.index[-1])
    grupos = dict(tuple(df.groupby("segunda")))

    for seg, r in g.iterrows():
        a_val = int(r["ano"])
        s_val = int(r["semana"])

        if a_val not in ia or s_val < 1 or s_val > 53:
            continue

        a_idx = ia[a_val]
        s_idx = s_val - 1

        # Verifica se a semana está fora dos limites de início/fim reais registrados para o ano
        limites_ano = g[g["ano"] == a_val].index
        fora_dos_limites = (seg < limites_ano.min()) or (seg > limites_ano.max())

        if r["gap"] or not (ini <= seg <= fim) or fora_dos_limites:
            cont[a_idx, s_idx] = np.nan
            texto[a_idx, s_idx] = f"<b>Semana {s_val}/{a_val}</b> ({seg:%d/%m})<br><i>sem dados de e-mail</i>"
        else:
            # Semana ativa com dados: se houver eventos, conta; se não, é estritamente 0.0
            d = grupos.get(seg)
            n = 0 if d is None else len(d)
            cont[a_idx, s_idx] = float(n)

            intervalo_texto = f"{seg:%d/%m}–{seg + pd.Timedelta(days=6):%d/%m}"
            texto[a_idx, s_idx] = f"<b>Semana {s_val}/{a_val}</b> ({intervalo_texto})<br><b>{n}</b> banca(s) real(is)" + _lista(d)

    return anos, cont, texto


def _suavizar(cont, k):
    """Calcula a média móvel sem deixar pontas flutuantes e respeitando NaNs."""
    if k <= 1:
        return cont
    out = np.full_like(cont, np.nan)
    for i, linha in enumerate(cont):
        s_linha = pd.Series(linha)
        # Roda a média móvel apenas onde temos valores válidos (incluindo os zeros que acabamos de fixar)
        m = s_linha.rolling(k, center=True, min_periods=1).mean().to_numpy()
        out[i] = np.where(np.isnan(linha), np.nan, m)
    return out


def _rotulo_ano(g, a):
    sem = g.loc[g["ano"] == a, "semana"]
    parcial = sem.min() > 1 or sem.max() < 52
    return f"{a} (parcial)" if parcial else str(a)

## 6. Desenhar as linhas
- `_tracos`: cria uma linha por ano (com tooltip) e, opcionalmente, a média dos anos tracejada;
- `_faixas_sem_dados`: pinta a faixa cinza onde não há e-mails (a legenda do ano aparece no topo da faixa).

In [19]:
def _tracos(df, g, periodo, suavizar, media, mostrar_legenda, cor_ano):
    anos, cont, texto = _semanal(df, g, periodo)
    liso = _suavizar(cont, suavizar)
    tr = []
    for k, a in enumerate(anos):
        t = []
        for s_idx, (x, v_suave) in enumerate(zip(texto[k], liso[k])):
            if x == "" or "<i>" in x:
                t.append(x)
            else:
                v_real = cont[k, s_idx]
                if suavizar > 1 and not np.isnan(v_suave):
                    # Torna visualmente indubitável que a altura do ponto é a média móvel e não o valor bruto real
                    txt_suave = (
                        f"<br>-----------------------------"
                        f"<br><b>Valor real desta semana:</b> {int(v_real)} banca(s)"
                        f"<br><b>Média móvel (altura plotada):</b> {v_suave:.2f} banca(s)"
                        f"<br><i>*Suavização de {suavizar} semanas ativa</i>"
                    )
                    # Removemos a menção duplicada para evitar confusão
                    base_text = x.split("<br><b>")[0] + f"<br>({x.split(" (")[1].split(")")[0]})"
                    t.append(base_text + txt_suave)
                else:
                    t.append(x)

        tr.append(go.Scatter(
            x=_SEMANAS, y=liso[k], mode="lines+markers", name=_rotulo_ano(g, a),
            legendgroup=str(a), showlegend=mostrar_legenda,
            line=dict(color=cor_ano[a], width=2, shape="linear"),
            marker=dict(size=5), connectgaps=False,
            text=t, hovertemplate="%{text}<extra></extra>"))

    if media:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            m = np.nanmean(liso, axis=0)
        n = np.isfinite(liso).sum(axis=0)
        m = np.where(n >= 2, m, np.nan)
        tr.append(go.Scatter(
            x=_SEMANAS, y=m, mode="lines", name="média dos anos", legendgroup="media",
            showlegend=mostrar_legenda, line=dict(color="#222", width=2.5, dash="dash"),
            connectgaps=False,
            text=[f"<b>Semana {s}</b><br>média dos anos: {v:.1f} bancas<br>({k} anos com dados)"
                  for s, v, k in zip(_SEMANAS, m, n)],
            hovertemplate="%{text}<extra></extra>"))
    return tr


def _faixas_sem_dados(fig, g, row=None, col=None):
    for a in sorted(g["ano"].unique()):
        sub = g[(g["ano"] == a) & g["gap"]]
        if len(sub):
            x0 = int(sub["semana"].min())
            x1 = int(sub["semana"].max())
            fig.add_vrect(
                x0=x0 - 0.5, x1=x1 + 0.5, fillcolor=CINZA, opacity=0.15,
                layer="below", line_width=0, row=row, col=col
            )

## 7. Função principal: `grafico`
Monta a figura completa. Parâmetros:
- `dividir`: `None`, `"programa"`, `"tipo"` ou `"nivel"`; cria um painel por categoria (as cores dos anos são as mesmas);
- `suavizar`: 1 (sem), 3 ou 5 semanas;
- `periodo`: tupla `(inicio, fim)` de segundas-feiras para recortar;
- `media`: mostra/oculta a média dos anos;
- `y_comum`: mesmo eixo Y em todos os painéis (comparar volumes) ou eixo próprio (comparar só a forma).

Interação (nativa do Plotly): clique na legenda liga/desliga um ano em todos os painéis, duplo clique isola um ano, arrastar faz zoom e o tooltip lista as bancas da semana.

In [20]:
def grafico(df, g, dividir=None, suavizar=1, periodo=None, media=True, y_comum=True, titulo=None):
    """dividir: None, 'programa', 'tipo' ou 'nivel' (um painel por categoria)."""
    cor_ano = {a: CORES_ANO[i % len(CORES_ANO)] for i, a in enumerate(sorted(g["ano"].unique()))}

    if not dividir:
        fig = go.Figure()
        tracos = _tracos(df, g, periodo, suavizar, media, True, cor_ano)
        for t in tracos:
            fig.add_trace(t)
        _faixas_sem_dados(fig, g)

        # Ajuste dinâmico de Y para painel único
        _, cont, _ = _semanal(df, g, periodo)
        liso = _suavizar(cont, suavizar)
        max_val = np.nanmax(liso) if np.any(np.isfinite(liso)) else 1
        teto_y = max(3, math.ceil(max_val * 1.15))

        fig.update_layout(
            title=titulo,
            xaxis=dict(tickmode="array", tickvals=_TICKS, ticktext=MESES, range=[0.5, 53.5], title="Semana do ano"),
            yaxis=dict(range=[0, teto_y], title="Nº de bancas na semana"),
            height=500, margin=dict(l=60, r=40, t=80, b=60), hovermode="closest",
            plot_bgcolor="#f8f9fa", paper_bgcolor="white"
        )
        return fig

    cats = sorted(df[dividir].dropna().unique())
    n_cats = len(cats)
    cols = 1 if n_cats <= 2 else 2
    rows = math.ceil(n_cats / cols)

    fig = make_subplots(
        rows=rows, cols=cols, subplot_titles=cats,
        shared_xaxes=True, shared_yaxes=y_comum,
        vertical_spacing=0.1, horizontal_spacing=0.08
    )

    # Calcular o valor máximo global para ajustar a escala do Y compartilhado ou individual
    max_global = 0.1
    for c in cats:
        sub_df = df[df[dividir] == c]
        _, cont, _ = _semanal(sub_df, g, periodo)
        liso = _suavizar(cont, suavizar)
        if np.any(np.isfinite(liso)):
            max_global = max(max_global, np.nanmax(liso))

    for idx, c in enumerate(cats):
        r_idx = (idx // cols) + 1
        c_idx = (idx % cols) + 1
        sub_df = df[df[dividir] == c]
        mostrar_leg = (idx == 0)
        tracos = _tracos(sub_df, g, periodo, suavizar, media, mostrar_leg, cor_ano)
        for t in tracos:
            fig.add_trace(t, row=r_idx, col=c_idx)
        _faixas_sem_dados(fig, g, row=r_idx, col=c_idx)

    # Determina o teto do eixo Y com folga de 15%
    teto_y = max(3, math.ceil(max_global * 1.15))

    fig.update_layout(
        title=titulo, height=330 * rows, margin=dict(l=60, r=40, t=80, b=60),
        hovermode="closest", plot_bgcolor="#f8f9fa", paper_bgcolor="white"
    )

    for r in range(1, rows + 1):
        for c in range(1, cols + 1):
            fig.update_xaxes(tickmode="array", tickvals=_TICKS, ticktext=MESES, range=[0.5, 53.5], row=r, col=c)
            fig.update_yaxes(range=[0, teto_y] if y_comum else [0, None], row=r, col=c)

    return fig

## 8. Painel interativo
`painel` junta tudo com controles (precisa de `ipywidgets`):
- caixas de seleção de **programa, tipo e nível** (filtros);
- **Dividir por**: nenhum, programa, tipo ou nível;
- **Suavizar**: sem, 3 ou 5 semanas;
- **Período**: slider com as semanas disponíveis (o recorte só esconde pontos, não muda o eixo);
- opções de mostrar a média dos anos e de usar o mesmo eixo Y nos painéis.

A cada mudança o gráfico é redesenhado; o título mostra quantas bancas passam pelos filtros.

In [21]:
def painel(caminho_csv, gap_min=6, manual=()):
    """Painel interativo sem opção de suavização. Requer ipywidgets."""
    import ipywidgets as W
    from IPython.display import clear_output, display
    from google.colab import output

    # Garante a ativação do gerenciador de widgets do Colab
    output.enable_custom_widget_manager()

    df = carregar(caminho_csv)
    g = montar_grade(df, gap_min, manual)

    def checks(titulo, opcoes):
        cbs = [W.Checkbox(value=True, description=str(o), indent=False,
                          layout=W.Layout(width="auto")) for o in opcoes]
        return W.VBox([W.HTML(f"<b>{titulo}</b>"), *cbs]), cbs

    box_p, cb_p = checks("Programa", df["sigla"].value_counts().index)
    box_t, cb_t = checks("Tipo", sorted(df["tipo"].unique()))
    box_n, cb_n = checks("Nível", sorted(df["nivel"].unique()))
    dividir = W.ToggleButtons(options=[("nenhum", None), ("programa", "programa"), ("tipo", "tipo"),
                                       ("nível", "nivel")], description="Dividir por")
    media = W.Checkbox(value=True, description="mostrar média dos anos", indent=False)
    ycom = W.Checkbox(value=True, description="mesmo eixo Y nos painéis", indent=False)
    slider = W.SelectionRangeSlider(
        options=[(f"{d:%d/%m/%Y}", d) for d in g.index], index=(0, len(g) - 1),
        description="Período", continuous_update=False,
        layout=W.Layout(width="95%"), style={"description_width": "60px"})

    # Criando container de saída e forçando a renderização do widget
    out = W.Output()

    def atualizar(*_):
        ps = [c.description for c in cb_p if c.value]
        ts = [c.description for c in cb_t if c.value]
        ns = [c.description for c in cb_n if c.value]
        ini, fim = slider.value
        d = df[df["sigla"].isin(ps) & df["tipo"].isin(ts) & df["nivel"].isin(ns)]
        n = d["segunda"].between(ini, fim).sum()
        # Chamar gráfico sem suavização (suavizar=1)
        fig = grafico(d, g, dividir.value, 1, (ini, fim), media.value, ycom.value,
                      titulo=f"Bancas por semana do ano, uma linha por ano · {n} bancas no filtro")
        with out:
            clear_output(wait=True)
            import plotly.graph_objects as go
            display(go.FigureWidget(fig))

    for w in cb_p + cb_t + cb_n + [slider, dividir, media, ycom]:
        w.observe(atualizar, "value")

    painel_completo = W.VBox([W.HBox([box_p, box_t, box_n, W.VBox([media, ycom])], layout=W.Layout(gap="40px")),
                             dividir, slider, out])

    display(painel_completo)
    atualizar()

### 8.1 Abrir o painel
Se o painel não aparecer (alguns ambientes, como JupyterLab sem a extensão de widgets), use as chamadas estáticas da seção 7, que não dependem de `ipywidgets`.

In [22]:
painel(CAMINHO)

## 9. Como interpretar e ajustar
- **Padrão que se repete:** linhas de anos diferentes com picos e vales nas mesmas semanas (ex.: pico no fim do 2º semestre).
- **Padrão que difere:** painéis (programas, tipos ou níveis) com formas diferentes entre si.
- **Cuidados:** 2024 e 2026 são anos parciais e março a maio de 2025 não tem e-mails; as faixas cinza e os "(parcial)" indicam isso. Com poucas bancas por semana, prefira suavizar e não tire conclusões de uma única semana.
- **Ajustes comuns:** `gap_min` e `manual` na seção 4; cores em `CORES_ANO` na seção 2; número de linhas do tooltip em `MAX_LINHAS_TOOLTIP`.